# New Dataset — Day-to-Night Synthetic Experiment

This notebook replicates the experimental pipeline of `ipcv-project-2-v2.ipynb` but uses the **New Dataset** (665 images, Pascal VOC XML annotations) instead of the IIT Madras / Final Dataset.

**Pipeline steps**

| Step | What happens |
|---|---|
| 0 | Setup, config, helpers |
| 1 | **Convert Pascal VOC → YOLO** format, verify labels |
| 2 | **EDA** of the converted New Dataset |
| 3 | **Leak-free train / val / test split** (perceptual-hash grouping, 70/15/15) |
| 4 | **Day-to-night transformation** (gamma darkening + CLAHE) to produce synthetic night images |
| 5 | Real night set preparation (same as original experiment) |
| 6 | Build the balanced benchmark (equal sizes, matched object-scale mix) |
| 7 | Train the grid: {raw, aligned} × {day, night-synthetic, night-real} × seeds |
| 8 | Evaluation tooling |
| 9 | Results — 3×3 transfer matrix, claim quantification |
| 10 | Diagnostics, qualitative results, saved artefacts |

**Claim under test:** Can day images from this independent dataset be transformed into synthetic night images that substitute for real night data?

Set `SMOKE_TEST = True` for a quick dry run of the whole pipeline.

## 0. Setup

In [ ]:
!pip install -q -U ultralytics

In [ ]:
import os, sys, re, json, glob, shutil, random, time, yaml, warnings, gc, zipfile
from pathlib import Path
from collections import Counter, defaultdict
import numpy as np
import pandas as pd
import cv2
import torch
import matplotlib.pyplot as plt
from PIL import Image as PILImage
from IPython.display import display, Image as IPyImage, FileLink
from tqdm.auto import tqdm
from ultralytics import YOLO

warnings.filterwarnings('ignore')

# ============================= CONFIG ==============================
SMOKE_TEST = False             # True = fast dry run

# ---- dataset paths -------------------------------------------------------
# New Dataset: flat VOC layout (images/ + annotations/)
NEW_DATA_DIR = Path('/kaggle/input/datasets/new-dataset-potholes')   # <-- set this to your Kaggle/Colab input path
# Real night dataset (same as original experiment)
REAL_DIR = Path('/kaggle/input/datasets/tavishisharma09/night-potholes-yolov8-format')
# Converted YOLO output will be written here (working directory)
YOLO_CONV_DIR = Path('/kaggle/working/new_dataset_yolo')
REAL_POTHOLE_CLASS_ID = 1     # pothole class id in the real night set

# ---- experiment grid -----------------------------------------------------
DOMAINS  = ['day', 'night', 'real']
PREPROCS = ['raw', 'aligned']
SEEDS    = [42, 43, 44]
SPLIT_FRACS = (0.70, 0.15, 0.15)   # train/val/test for the new dataset

# ---- image processing ('aligned') ----------------------------------------
TARGET_MEAN = 90
GAMMA_MIN   = 0.4
CLAHE_CLIP  = 2.0

# ---- near-duplicate detection --------------------------------------------
HASH_THR    = 6
CORR_MIN    = 0.70
CORR_MARGIN = 0.10
BOX_IOU_MIN = 0.70
SIZE_BINS   = (0.005, 0.02, 0.08)

# ---- training ------------------------------------------------------------
MODEL_WEIGHTS = 'yolo11m.pt'
IMGSZ, BATCH, EPOCHS, PATIENCE, CLOSE_MOSAIC = 640, 16, 60, 15, 8
RETRAIN = False
TIME_BUDGET_H = 9.5
EVAL_BATCH = 16
N_BOOT = 300
RUN_IMGSZ_SWEEP = True
N_CAP = None
SEED = 42

if SMOKE_TEST:
    IMGSZ, BATCH, EPOCHS, PATIENCE, CLOSE_MOSAIC = 320, 8, 2, 5, 1
    SEEDS, N_CAP, N_BOOT = [42], 40, 50

WORK = Path('/kaggle/working')
RUNS = WORK / 'runs' / ('smoke' if SMOKE_TEST else 'main')
BENCH = WORK / ('bench_smoke' if SMOKE_TEST else 'bench')
FIGS  = WORK / 'figures'
for d in (RUNS, BENCH, FIGS, YOLO_CONV_DIR):
    d.mkdir(parents=True, exist_ok=True)

DEVICE = 0 if torch.cuda.is_available() else 'cpu'
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)

TRAIN_KW = dict(
    epochs=EPOCHS, patience=PATIENCE, imgsz=IMGSZ, batch=BATCH,
    optimizer='auto', cos_lr=True, close_mosaic=CLOSE_MOSAIC,
    warmup_epochs=3, mosaic=1.0, mixup=0.0, scale=0.7,
    fliplr=0.5, flipud=0.0, degrees=0.0,
    hsv_h=0.015, hsv_s=0.6, hsv_v=0.4,
    workers=2, device=DEVICE, plots=False,
    cache='ram', amp=True, deterministic=True, exist_ok=True
)
print('device:', DEVICE, '| torch', torch.__version__, '| smoke:', SMOKE_TEST)

In [ ]:
IMAGE_EXTS = {'.jpg', '.jpeg', '.png', '.bmp', '.webp'}

def list_images(d):
    return sorted(p for p in Path(d).iterdir() if p.suffix.lower() in IMAGE_EXTS)

def label_path_for(img_path):
    img_path = Path(img_path)
    return img_path.parent.parent / 'labels' / f'{img_path.stem}.txt'

def read_yolo_labels(path, keep_ids=None):
    rows = []
    if path is not None and Path(path).exists():
        for line in Path(path).read_text().splitlines():
            p = line.split()
            if len(p) < 5: continue
            v = [float(x) for x in p]
            if len(v) == 5:
                c, xc, yc, w, h = v
            else:
                xs, ys = v[1::2], v[2::2]
                x1, x2, y1, y2 = min(xs), max(xs), min(ys), max(ys)
                c, xc, yc, w, h = v[0], (x1+x2)/2, (y1+y2)/2, x2-x1, y2-y1
            rows.append([c, xc, yc, w, h])
    lab = np.array(rows, dtype=float).reshape(-1, 5)
    if keep_ids is not None:
        lab = lab[np.isin(lab[:, 0].astype(int), keep_ids)]
    return lab

def xywhn_to_xyxyn(lab):
    xc, yc, w, h = lab[:,1], lab[:,2], lab[:,3], lab[:,4]
    return np.stack([xc-w/2, yc-h/2, xc+w/2, yc+h/2], 1)

def iou_matrix(a, b):
    if len(a) == 0 or len(b) == 0:
        return np.zeros((len(a), len(b)))
    ix1 = np.maximum(a[:,None,0], b[None,:,0]); iy1 = np.maximum(a[:,None,1], b[None,:,1])
    ix2 = np.minimum(a[:,None,2], b[None,:,2]); iy2 = np.minimum(a[:,None,3], b[None,:,3])
    inter = np.clip(ix2-ix1,0,None) * np.clip(iy2-iy1,0,None)
    aa = (a[:,2]-a[:,0])*(a[:,3]-a[:,1]); ab = (b[:,2]-b[:,0])*(b[:,3]-b[:,1])
    return inter / (aa[:,None] + ab[None,:] - inter + 1e-9)

def mean_gray(path):
    img = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    return float(img.mean()) if img is not None else 0.0

def draw_boxes(img_bgr, lab, color=(0,255,0), thick=3):
    h, w = img_bgr.shape[:2]
    for _, xc, yc, bw, bh in lab:
        cv2.rectangle(img_bgr,
            (int((xc-bw/2)*w), int((yc-bh/2)*h)),
            (int((xc+bw/2)*w), int((yc+bh/2)*h)), color, thick)
    return img_bgr

def savefig(name):
    plt.savefig(FIGS / name, dpi=150, bbox_inches='tight')

# --- perceptual hash helpers -----------------------------------------------
def dhash_pair(path):
    g = cv2.imread(str(path), cv2.IMREAD_GRAYSCALE)
    g = cv2.resize(g, (9,8), interpolation=cv2.INTER_AREA)
    h  = (g[:,1:] > g[:,:-1]).ravel()
    gf = g[:,::-1]
    hf = (gf[:,1:] > gf[:,:-1]).ravel()
    return h, hf

def hash_arrays(paths):
    if not paths:
        return np.zeros((0,64),bool), np.zeros((0,64),bool)
    hs, hfs = zip(*[dhash_pair(p) for p in paths])
    return np.array(hs,dtype=bool).reshape(len(paths),64), np.array(hfs,dtype=bool).reshape(len(paths),64)

def min_hamming(Ha, Hb, Hb_flip, chunk=128):
    best_d = np.full(len(Ha), 64); best_i = np.full(len(Ha), -1)
    if len(Hb) == 0:
        return best_d, best_i
    for s in range(0, len(Ha), chunk):
        a = Ha[s:s+chunk, None, :]
        d = np.minimum((a != Hb[None]).sum(2), (a != Hb_flip[None]).sum(2))
        best_i[s:s+chunk] = d.argmin(1); best_d[s:s+chunk] = d.min(1)
    return best_d, best_i

_trapz = np.trapezoid if hasattr(np, 'trapezoid') else np.trapz
print('Helpers loaded.')

## 1. Convert Pascal VOC XML → YOLO Format

The New Dataset uses Pascal VOC XML annotations (`<xmin>/<ymin>/<xmax>/<ymax>` in pixel coords). We convert every annotation to YOLO normalised format (`class xc yc w h`) and write `.txt` files alongside the images in a YOLO-style folder structure under `YOLO_CONV_DIR`.

Only `pothole` objects are kept (class id remapped to 0).

In [ ]:
import xml.etree.ElementTree as ET

# Output YOLO layout: YOLO_CONV_DIR/images/*.png  +  YOLO_CONV_DIR/labels/*.txt
yolo_img_dir = YOLO_CONV_DIR / 'images'
yolo_lbl_dir = YOLO_CONV_DIR / 'labels'
yolo_img_dir.mkdir(parents=True, exist_ok=True)
yolo_lbl_dir.mkdir(parents=True, exist_ok=True)

ANNO_DIR = NEW_DATA_DIR / 'annotations'
IMG_DIR  = NEW_DATA_DIR / 'images'

def convert_voc_xml(xml_path):
    """Returns list of YOLO lines (class 0, normalised xc yc w h) for one XML."""
    tree = ET.parse(xml_path)
    root = tree.getroot()
    sz  = root.find('size')
    img_w = float(sz.find('width').text)
    img_h = float(sz.find('height').text)
    lines = []
    for obj in root.findall('object'):
        name = obj.find('name').text.strip().lower()
        if name != 'pothole':
            continue
        bb = obj.find('bndbox')
        xmin = float(bb.find('xmin').text)
        ymin = float(bb.find('ymin').text)
        xmax = float(bb.find('xmax').text)
        ymax = float(bb.find('ymax').text)
        # clamp to image
        xmin = max(0, min(xmin, img_w)); xmax = max(0, min(xmax, img_w))
        ymin = max(0, min(ymin, img_h)); ymax = max(0, min(ymax, img_h))
        if xmax <= xmin or ymax <= ymin:
            continue
        xc = ((xmin + xmax) / 2) / img_w
        yc = ((ymin + ymax) / 2) / img_h
        bw = (xmax - xmin) / img_w
        bh = (ymax - ymin) / img_h
        lines.append(f'0 {xc:.6f} {yc:.6f} {bw:.6f} {bh:.6f}')
    return lines

# collect all XMLs
xml_files = sorted(ANNO_DIR.glob('*.xml'))
print(f'Found {len(xml_files)} XML annotation files')

converted, skipped_no_boxes, skipped_no_img = 0, 0, 0
all_items = []   # (img_path, label_path) pairs

for xml_path in tqdm(xml_files, desc='Converting VOC → YOLO'):
    # find the matching image
    stem = xml_path.stem   # e.g. 'potholes0'
    img_path = None
    for ext in IMAGE_EXTS:
        candidate = IMG_DIR / (stem + ext)
        if candidate.exists():
            img_path = candidate
            break
    if img_path is None:
        skipped_no_img += 1
        continue

    lines = convert_voc_xml(xml_path)
    if not lines:
        skipped_no_boxes += 1
        continue

    # copy image to YOLO layout
    dst_img = yolo_img_dir / img_path.name
    dst_lbl = yolo_lbl_dir / (stem + '.txt')
    if not dst_img.exists():
        shutil.copy2(img_path, dst_img)
    dst_lbl.write_text('\n'.join(lines))
    all_items.append({'img': dst_img, 'lbl': dst_lbl})
    converted += 1

print(f'Converted : {converted}')
print(f'Skipped (no image found) : {skipped_no_img}')
print(f'Skipped (zero pothole boxes) : {skipped_no_boxes}')
print(f'Total usable items : {len(all_items)}')

In [ ]:
# Quick sanity-check: show 4 converted images with boxes drawn
rng = random.Random(SEED)
show_items = rng.sample(all_items, min(4, len(all_items)))
fig, axes = plt.subplots(1, len(show_items), figsize=(5*len(show_items), 4))
for ax, it in zip(axes, show_items):
    img = cv2.imread(str(it['img']))
    lab = read_yolo_labels(it['lbl'])
    img = draw_boxes(img, lab)
    ax.imshow(cv2.cvtColor(img, cv2.COLOR_BGR2RGB))
    ax.set_title(f"{it['img'].name}\n{len(lab)} boxes", fontsize=8)
    ax.axis('off')
plt.suptitle('Converted YOLO labels — sanity check')
plt.tight_layout(); savefig('voc2yolo_check.png'); plt.show()
print('Total boxes in converted set:',
      sum(len(read_yolo_labels(it['lbl'])) for it in all_items))

## 2. EDA of the New Dataset

Check class distribution, boxes per image, bounding-box size distribution, image resolutions, and brightness — the same diagnostics as the original notebook.

In [ ]:
imgs  = [it['img'] for it in all_items]
labels = [read_yolo_labels(it['lbl']) for it in all_items]

boxes_per_img = [len(l) for l in labels]
areas = np.concatenate([l[:,3]*l[:,4]*100 for l in labels if len(l)])
brightness = [mean_gray(p) for p in imgs]
sizes = []
for p in imgs:
    with PILImage.open(p) as im:
        sizes.append(im.size)
size_ctr = Counter(sizes)

print('=== New Dataset EDA ===')
print(f'Images         : {len(imgs)}')
print(f'Total boxes    : {sum(boxes_per_img)}')
print(f'Boxes/image    : mean={np.mean(boxes_per_img):.2f}  median={np.median(boxes_per_img):.0f}  max={max(boxes_per_img)}')
print(f'Box area (% of img): min={np.nanmin(areas):.4f}  median={np.nanmedian(areas):.3f}  max={np.nanmax(areas):.2f}')
print(f'Mean brightness: {np.mean(brightness):.1f}')
print(f'Top image sizes: {size_ctr.most_common(5)}')

fig, axes = plt.subplots(1, 3, figsize=(15, 4))
axes[0].hist(boxes_per_img, bins=range(0, max(boxes_per_img)+2), rwidth=0.8)
axes[0].set_title('Boxes per image'); axes[0].set_xlabel('count')
axes[1].hist(np.log10(areas + 1e-6), bins=40)
axes[1].set_title('log10 box area (% of image)')
axes[2].hist(brightness, bins=30)
axes[2].set_title('Mean brightness (grey level)')
plt.suptitle('New Dataset EDA')
plt.tight_layout(); savefig('eda_new_dataset.png'); plt.show()

## 3. Leak-Free Train / Val / Test Split

Because the New Dataset has **no pre-existing splits**, we apply the same perceptual-hash grouping used for the real night set:
- Images are grouped by near-duplicate hash (Hamming ≤ `HASH_THR`, including mirrors).
- Whole groups are assigned to one split, so no near-duplicate can appear in both train and test.
- Sizes: 70 / 15 / 15 %

In [ ]:
def split_by_hash(items, fracs, hash_thr, seed):
    """Returns (assign_lists, groups). assign_lists[i] = list of indices for split i."""
    paths = [it['img'] for it in items]
    H, Hf = hash_arrays(paths)
    n = len(items)
    parent = list(range(n))

    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x

    for i in range(n):
        dm = np.minimum((H[i][None] != H).sum(1), (H[i][None] != Hf).sum(1))
        for j in np.where(dm <= hash_thr)[0]:
            ri, rj = find(i), find(int(j))
            if ri != rj:
                parent[ri] = rj

    groups = defaultdict(list)
    for i in range(n):
        groups[find(i)].append(i)
    glist = list(groups.values())
    random.Random(seed).shuffle(glist)

    target = [f * n for f in fracs]
    cur = [0, 0, 0]
    assign = [[], [], []]
    for g in sorted(glist, key=len, reverse=True):
        k = int(np.argmax([target[s] - cur[s] for s in range(3)]))
        assign[k] += g; cur[k] += len(g)
    return assign, glist

print('Computing perceptual hashes (this may take ~30s)...')
assign, glist = split_by_hash(all_items, SPLIT_FRACS, HASH_THR, SEED)

split_names = ['train', 'val', 'test']
day_split = {sp: [all_items[i] for i in idx]
             for sp, idx in zip(split_names, assign)}

sizes_ = sorted((len(g) for g in glist), reverse=True)
print(f'{len(all_items)} images → {len(glist)} near-dup groups')
print(f'Largest groups: {sizes_[:6]} | Singletons: {sum(s==1 for s in sizes_)}')
for sp, its in day_split.items():
    n_boxes = sum(len(read_yolo_labels(it['lbl'])) for it in its)
    print(f'  {sp:5s}: {len(its):4d} images, {n_boxes} boxes')

# Audit: no near-duplicate should cross train ↔ val/test
Htr, Htr_f = hash_arrays([it['img'] for it in day_split['train']])
for sp in ('val', 'test'):
    Hs, _ = hash_arrays([it['img'] for it in day_split[sp]])
    dmin, _ = min_hamming(Hs, Htr, Htr_f)
    leaky = int((dmin <= HASH_THR).sum())
    print(f'Cross-split duplicates {sp}→train: {leaky} (should be 0)')

## 4. Day-to-Night Transformation

We apply the **same gamma-lift + CLAHE pipeline** as `night_sim.ipynb` to every day image to create synthetic night images. The labels are **shared identically** between the day and night-synthetic domains (same boxes, same scene — only the pixel values differ).

This means the Day model and the Night-synthetic model see:
- **Identical scenes and bounding boxes**
- **Different pixel values only** (the transformation is the only variable)

This controlled setup lets us isolate the effect of the transformation.

In [ ]:
_CLAHE = cv2.createCLAHE(clipLimit=CLAHE_CLIP, tileGridSize=(8, 8))

W_BGR = np.array([0.114, 0.587, 0.299], np.float32)  # BGR→grey weights

def simulate_night(img_bgr,
                   dark_factor=0.35,
                   noise_std=8,
                   vignette_strength=0.45,
                   color_shift=(-10, -5, +5),
                   gamma=2.0,
                   seed=None):
    """
    Convert a daytime BGR image to a synthetic night image.
    Steps:
      1. Gamma darkening (gamma > 1 = darken)
      2. Per-channel colour shift (road lighting is typically warm)
      3. Gaussian noise (sensor noise at high ISO)
      4. Radial vignette (headlight fall-off)
    """
    rng = np.random.default_rng(seed)
    img = img_bgr.astype(np.float32) / 255.0

    # 1. Gamma darken
    img = np.power(np.clip(img, 1e-6, 1.0), gamma) * dark_factor

    # 2. Colour shift (B, G, R offsets in [0,255] space)
    for c, shift in enumerate(color_shift):
        img[:,:,c] = np.clip(img[:,:,c] + shift/255.0, 0, 1)

    # 3. Gaussian noise
    noise = rng.normal(0, noise_std/255.0, img.shape).astype(np.float32)
    img = np.clip(img + noise, 0, 1)

    # 4. Vignette
    h, w = img.shape[:2]
    Y, X = np.ogrid[:h, :w]
    cx, cy = w/2, h/2
    dist = np.sqrt(((X-cx)/cx)**2 + ((Y-cy)/cy)**2)
    vignette = np.clip(1.0 - vignette_strength * dist, 0, 1).astype(np.float32)
    img = img * vignette[:,:,None]

    return (np.clip(img, 0, 1) * 255).astype(np.uint8)

def align_image(img_bgr):
    """Gamma-lift + CLAHE for the 'aligned' preprocessing condition."""
    grey = cv2.cvtColor(img_bgr, cv2.COLOR_BGR2GRAY).astype(np.float32) / 255
    m = float(np.clip(grey.mean(), 1e-3, 0.999))
    t = TARGET_MEAN / 255
    gamma_val = float(np.clip(np.log(t) / np.log(m), GAMMA_MIN, 1.0))
    lut = (255 * (np.arange(256)/255.0)**gamma_val).astype(np.uint8)
    img = cv2.LUT(img_bgr, lut)
    l, a, b = cv2.split(cv2.cvtColor(img, cv2.COLOR_BGR2LAB))
    return cv2.cvtColor(cv2.merge((_CLAHE.apply(l), a, b)), cv2.COLOR_LAB2BGR)

# --- Demo: show 3 day images and their synthetic night counterparts ---
rng = random.Random(SEED)
demo_items = rng.sample(day_split['train'], min(3, len(day_split['train'])))
fig, axes = plt.subplots(3, 3, figsize=(15, 12))
for i, it in enumerate(demo_items):
    img_day = cv2.imread(str(it['img']))
    lab = read_yolo_labels(it['lbl'])
    img_night = simulate_night(img_day, seed=SEED+i)
    img_aligned = align_image(img_night)
    for j, (title, im) in enumerate([
        (f"Day (mean={mean_gray(it['img']):.0f})", img_day),
        ('Synthetic Night (raw)', img_night),
        ('Synthetic Night (aligned)', img_aligned)
    ]):
        ax = axes[i][j]
        ax.imshow(cv2.cvtColor(draw_boxes(im.copy(), lab), cv2.COLOR_BGR2RGB))
        ax.set_title(title, fontsize=9); ax.axis('off')
plt.suptitle('Day → Night Transformation Demo (New Dataset)')
plt.tight_layout(); savefig('night_transform_demo.png'); plt.show()

## 5. Real Night Set — Class Filtering & Leak-Free Split

Same as the original experiment: load the real night set, keep only pothole boxes (class id `REAL_POTHOLE_CLASS_ID`), remap to class 0, then split into train/val/test grouped by near-duplicate hash.

In [ ]:
def find_dataset_root(base):
    base = Path(base)
    if not base.exists():
        raise FileNotFoundError(f'{base} not found')
    lvl1 = [p for p in base.iterdir() if p.is_dir()]
    lvl2 = [q for p in lvl1 for q in p.iterdir() if q.is_dir()]
    for c in [base] + lvl1 + lvl2:
        if any((c / s / 'images').is_dir() for s in ('train','valid','val','test')):
            return c
    raise FileNotFoundError(f'No YOLO split folders found under {base}')

def split_dirs(root):
    out = {}
    for key, names in {'train':['train'],'val':['valid','val'],'test':['test']}.items():
        for n in names:
            if (root / n / 'images').is_dir():
                out[key] = (root / n / 'images', root / n / 'labels')
                break
    return out

REAL_POTHOLE_IDS = [int(REAL_POTHOLE_CLASS_ID)]
real_root = find_dataset_root(REAL_DIR)
real_sd   = split_dirs(real_root)
print('Real root:', real_root, '| splits:', list(real_sd))

real_items = []
skipped_real = dropped_boxes_real = 0
for sp, (idir, ldir) in real_sd.items():
    for p in list_images(idir):
        lp = ldir / f'{p.stem}.txt'
        lab = read_yolo_labels(lp)
        keep = np.isin(lab[:,0].astype(int), REAL_POTHOLE_IDS)
        dropped_boxes_real += int((~keep).sum())
        if keep.sum() == 0:
            skipped_real += 1; continue
        lab = lab[keep].copy(); lab[:,0] = 0
        real_items.append({'path': p, 'lab': lab})

print(f'Real images kept: {len(real_items)}')
print(f'Images without potholes skipped: {skipped_real}')
print(f'Non-pothole boxes dropped: {dropped_boxes_real}')
print(f'Total pothole boxes: {sum(len(r["lab"]) for r in real_items)}')

In [ ]:
def split_real(items, fracs, hash_thr, seed):
    paths = [it['path'] for it in items]
    H, Hf = hash_arrays(paths)
    n = len(items)
    parent = list(range(n))
    def find(x):
        while parent[x] != x:
            parent[x] = parent[parent[x]]; x = parent[x]
        return x
    for i in range(n):
        dm = np.minimum((H[i][None] != H).sum(1), (H[i][None] != Hf).sum(1))
        for j in np.where(dm <= hash_thr)[0]:
            ri, rj = find(i), find(int(j))
            if ri != rj: parent[ri] = rj
    groups = defaultdict(list)
    for i in range(n): groups[find(i)].append(i)
    glist = list(groups.values())
    random.Random(seed).shuffle(glist)
    target = [f*n for f in fracs]; cur = [0,0,0]; assign = [[],[],[]]
    for g in sorted(glist, key=len, reverse=True):
        k = int(np.argmax([target[s]-cur[s] for s in range(3)]))
        assign[k] += g; cur[k] += len(g)
    return assign, glist

assign_r, glist_r = split_real(real_items, SPLIT_FRACS, HASH_THR, SEED)
real_split = {sp: [real_items[i] for i in idx]
              for sp, idx in zip(('train','val','test'), assign_r)}

for sp, v in real_split.items():
    print(f'real {sp:5s}: {len(v):4d} images, {sum(len(r["lab"]) for r in v)} boxes')

# leak audit
Htr_r, Htr_rf = hash_arrays([r['path'] for r in real_split['train']])
for sp in ('val','test'):
    Hs, _ = hash_arrays([r['path'] for r in real_split[sp]])
    dmin, _ = min_hamming(Hs, Htr_r, Htr_rf)
    print(f'real {sp}: near-dups in train = {int((dmin <= HASH_THR).sum())} (should be 0)')

## 6. Build the Balanced Benchmark

All three domains (day, night-synthetic, night-real) are balanced to the same number of images per split. The day and night-synthetic domains use **identical images and labels** (same scene, same boxes — only pixel values differ). The real domain is size-matched.

All images are decoded and re-encoded at JPEG q95 to remove any compression-domain advantage.

In [ ]:
import hashlib

nb_bins = len(SIZE_BINS) + 1

def size_bin(lab):
    return 0 if len(lab) == 0 else int(np.searchsorted(SIZE_BINS, (lab[:,3]*lab[:,4]).max()))

def stratified_pick(bins, target_props, n, rng):
    byb = {b: [i for i, x in enumerate(bins) if x == b] for b in range(nb_bins)}
    for v in byb.values(): rng.shuffle(v)
    picked = []
    for b in range(nb_bins):
        picked += byb[b][:int(round(n * target_props.get(b, 0)))]
    if len(picked) > n:
        rng.shuffle(picked); picked = picked[:n]
    elif len(picked) < n:
        taken = set(picked); rest = [i for i in range(len(bins)) if i not in taken]
        rng.shuffle(rest); picked += rest[:n-len(picked)]
    return sorted(picked)

rng_b = random.Random(SEED)
cap_tv = None if N_CAP is None else max(10, N_CAP // 2)
N_TRAIN = min(len(day_split['train']), len(real_split['train']), N_CAP or 10**9)
N_VAL   = min(len(day_split['val']),   len(real_split['val']),   cap_tv or 10**9)
N_TEST  = min(len(day_split['test']),  len(real_split['test']),  cap_tv or 10**9)
print(f'Balanced sizes → train {N_TRAIN} | val {N_VAL} | test {N_TEST} images per domain')

# Real selection
sel_real = {
    'train': sorted(rng_b.sample(range(len(real_split['train'])), N_TRAIN)),
    'val':   sorted(rng_b.sample(range(len(real_split['val'])),   N_VAL)),
    'test':  sorted(rng_b.sample(range(len(real_split['test'])),  N_TEST))
}
real_sel = {sp: [real_split[sp][i] for i in idx] for sp, idx in sel_real.items()}

# Day/Night-synthetic selection (stratified on object size)
real_bins_tr = [size_bin(r['lab']) for r in real_sel['train']]
target_props = {b: real_bins_tr.count(b)/len(real_bins_tr) for b in range(nb_bins)}
pool_bins    = [size_bin(read_yolo_labels(it['lbl'])) for it in day_split['train']]
idx_tr = stratified_pick(pool_bins, target_props, N_TRAIN, rng_b)

pair_sel = {
    'train': [day_split['train'][i] for i in idx_tr],
    'val':   [day_split['val'][i]   for i in sorted(rng_b.sample(range(len(day_split['val'])),  N_VAL))],
    'test':  [day_split['test'][i]  for i in sorted(rng_b.sample(range(len(day_split['test'])), N_TEST))]
}

got = Counter(size_bin(read_yolo_labels(it['lbl'])) for it in pair_sel['train'])
print('Object-size bin distribution (train):')
print(pd.DataFrame({'real': target_props, 'day+night': {b: got.get(b,0)/N_TRAIN for b in range(nb_bins)}}).round(3).T.to_string())

In [ ]:
def write_set(pre, dom, split, entries):
    """
    entries: list of dict(name, src_img, lab, apply_night)
    Writes preprocessed JPEGs + class-0 labels; uses a content hash to avoid re-writing.
    """
    root = BENCH / pre / dom / split
    sig = hashlib.md5(
        (f'{pre}|{TARGET_MEAN}|{GAMMA_MIN}|{CLAHE_CLIP}|' +
         '|'.join(f"{e['name']}:{e['src_img']}" for e in entries)
        ).encode()).hexdigest()
    marker = root / '.done'
    idir, ldir = root / 'images', root / 'labels'
    if not (marker.exists() and marker.read_text() == sig):
        if root.exists(): shutil.rmtree(root)
        idir.mkdir(parents=True); ldir.mkdir(parents=True)
        for e in tqdm(entries, desc=f'{pre}/{dom}/{split}', leave=False):
            img = cv2.imread(str(e['src_img']))
            if img is None:
                img = np.zeros((320,320,3), np.uint8)
            if e.get('apply_night', False):
                img = simulate_night(img, seed=SEED)
            if pre == 'aligned':
                img = align_image(img)
            cv2.imwrite(str(idir / f"{e['name']}.jpg"), img,
                        [cv2.IMWRITE_JPEG_QUALITY, 95])
            with open(ldir / f"{e['name']}.txt", 'w') as f:
                for _, xc, yc, w, h in e['lab']:
                    f.write(f'0 {xc:.6f} {yc:.6f} {w:.6f} {h:.6f}\n')
        marker.write_text(sig)
    return list_images(idir)

BENCH_SETS = {}
for pre in PREPROCS:
    for split in ('train','val','test'):
        # Day domain: original daytime images
        day_entries = [{'name': f'day_{split}_{k:05d}',
                        'src_img': it['img'],
                        'lab': read_yolo_labels(it['lbl']),
                        'apply_night': False}
                       for k, it in enumerate(pair_sel[split])]
        # Night-synthetic domain: same images, night transformation applied
        night_entries = [{'name': f'night_{split}_{k:05d}',
                          'src_img': it['img'],
                          'lab': read_yolo_labels(it['lbl']),
                          'apply_night': True}
                         for k, it in enumerate(pair_sel[split])]
        # Real domain
        real_entries = [{'name': f'real_{split}_{k:05d}',
                         'src_img': r['path'],
                         'lab': r['lab'],
                         'apply_night': False}
                        for k, r in enumerate(real_sel[split])]
        for dom, entries in [('day', day_entries), ('night', night_entries), ('real', real_entries)]:
            BENCH_SETS[(pre, dom, split)] = {'images': write_set(pre, dom, split, entries)}

# Write data.yaml files
YAMLS = {}
for pre in PREPROCS:
    for dom in DOMAINS:
        yp = BENCH / f'{dom}_{pre}.yaml'
        with open(yp, 'w') as f:
            yaml.dump({'train': str(BENCH / pre / dom / 'train' / 'images'),
                       'val':   str(BENCH / pre / dom / 'val'   / 'images'),
                       'test':  str(BENCH / pre / dom / 'test'  / 'images'),
                       'nc': 1, 'names': {0: 'pothole'}}, f, sort_keys=False)
        YAMLS[(pre, dom)] = yp
print('Benchmark ready:', BENCH)

In [ ]:
# Confirm balance across domains
rows = []
for dom in DOMAINS:
    for split in ('train','val','test'):
        imgs_raw = BENCH_SETS[('raw', dom, split)]['images']
        labs = [read_yolo_labels(label_path_for(p)) for p in imgs_raw]
        areas = np.concatenate([l[:,3]*l[:,4]*100 for l in labs if len(l)])
        row = {'domain': dom, 'split': split, 'images': len(imgs_raw),
               'boxes': len(areas), 'boxes/img': round(len(areas)/len(imgs_raw),2),
               'median box %': round(float(np.median(areas)),3)}
        for pre in PREPROCS:
            sample = random.Random(SEED).sample(BENCH_SETS[(pre, dom, split)]['images'], min(100, len(imgs_raw)))
            row[f'brightness[{pre}]'] = round(float(np.mean([mean_gray(p) for p in sample])),1)
        rows.append(row)
bal = pd.DataFrame(rows).set_index(['domain','split'])
display(bal)

fig, ax = plt.subplots(1, len(PREPROCS)+1, figsize=(6*(len(PREPROCS)+1), 4))
for k, pre in enumerate(PREPROCS):
    for dom in DOMAINS:
        ax[k].hist([mean_gray(p) for p in BENCH_SETS[(pre,dom,'train')]['images']],
                   bins=30, alpha=0.5, density=True, label=dom)
    ax[k].set_title(f'Train brightness [{pre}]'); ax[k].legend()
for dom in DOMAINS:
    a = np.concatenate([read_yolo_labels(label_path_for(p))[:,3]*read_yolo_labels(label_path_for(p))[:,4]*100
                        for p in BENCH_SETS[('raw',dom,'train')]['images'] if len(read_yolo_labels(label_path_for(p)))])
    ax[-1].hist(np.log10(a+1e-6), bins=40, alpha=0.5, density=True, label=dom)
ax[-1].set_title('log10 box area'); ax[-1].legend()
plt.tight_layout(); savefig('benchmark_balance.png'); plt.show()

## 7. Train the Grid

For every `seed × preprocessing × domain`, a fresh COCO-pretrained YOLO11m is trained with **identical hyperparameters**. Only the training images differ:
- **Day**: original daytime images from the New Dataset
- **Night-synthetic**: same images after the gamma-darkening + noise + vignette transformation
- **Night-real**: real night images (independent dataset)

Runs are cached — if `best.pt` already exists the run is skipped (safe to re-run after a timeout).

In [ ]:
def run_name(pre, dom, seed): return f'{dom}_{pre}_s{seed}'
def best_path(pre, dom, seed): return RUNS / run_name(pre, dom, seed) / 'weights' / 'best.pt'

plan = [(seed, pre, dom) for seed in SEEDS for pre in PREPROCS for dom in DOMAINS]
t_start = time.time()
for seed, pre, dom in plan:
    if best_path(pre, dom, seed).exists() and not RETRAIN:
        print(f'[skip] {run_name(pre, dom, seed)} already trained'); continue
    if (time.time() - t_start) / 3600 > TIME_BUDGET_H:
        print(f'Time budget {TIME_BUDGET_H}h reached. Re-run cell to continue.'); break
    t0 = time.time()
    model = YOLO(MODEL_WEIGHTS)
    model.train(
        data=str(YAMLS[(pre, dom)]),
        seed=seed,
        project=str(RUNS),
        name=run_name(pre, dom, seed),
        **TRAIN_KW
    )
    print(f'{run_name(pre, dom, seed)} done in {(time.time()-t0)/60:.1f} min')
    del model; gc.collect(); torch.cuda.empty_cache()

done = [(s,p,d) for s,p,d in plan if best_path(p,d,s).exists()]
print(f'Trained runs: {len(done)} / {len(plan)}')

In [ ]:
fig, ax = plt.subplots(1, len(PREPROCS), figsize=(7*len(PREPROCS), 4), squeeze=False)
cols = {'day': '#c9a227', 'night': '#3b5ba9', 'real': '#3a8f5c'}
for k, pre in enumerate(PREPROCS):
    for dom in DOMAINS:
        f = RUNS / run_name(pre, dom, SEEDS[0]) / 'results.csv'
        if f.exists():
            df_ = pd.read_csv(f); df_.columns = [c.strip() for c in df_.columns]
            ax[0][k].plot(df_['epoch'], df_['metrics/mAP50(B)'],
                         color=cols[dom], label=f'{dom}-trained')
    ax[0][k].set_title(f'Own-domain val mAP@0.5 [{pre}], seed {SEEDS[0]}')
    ax[0][k].set_xlabel('epoch'); ax[0][k].legend()
plt.tight_layout(); savefig('training_curves.png'); plt.show()

## 8. Evaluation Tooling

Model-agnostic evaluator (greedy IoU matching, 101-point AP). Each model's confidence threshold is chosen on **its own training-domain validation set** — the same fair protocol as the original notebook.

In [ ]:
IOU_THRS = np.linspace(0.5, 0.95, 10)

def gt_to_xyxy(lab, shape):
    h, w = shape
    xc, yc, bw, bh = lab[:,1]*w, lab[:,2]*h, lab[:,3]*w, lab[:,4]*h
    return np.stack([xc-bw/2, yc-bh/2, xc+bw/2, yc+bh/2], 1)

def predict_dataset(model, img_paths, conf=0.001, iou=0.7, imgsz=None):
    out = []
    for i in range(0, len(img_paths), EVAL_BATCH):
        chunk = [str(p) for p in img_paths[i:i+EVAL_BATCH]]
        for r in model.predict(chunk, conf=conf, iou=iou,
                               imgsz=imgsz or IMGSZ, max_det=300,
                               device=DEVICE, verbose=False):
            b = r.boxes
            out.append({'xyxy': b.xyxy.cpu().numpy(),
                        'conf': b.conf.cpu().numpy(),
                        'shape': r.orig_shape})
    return out

def match_image(xyxy, conf, g, thr):
    order = np.argsort(-conf)
    tp = np.zeros(len(order), bool); gt_conf = -np.ones(len(g))
    if len(order) and len(g):
        ious = iou_matrix(xyxy[order], g); taken = np.zeros(len(g), bool)
        for k in range(len(order)):
            cand = np.where(taken, -1.0, ious[k]); j = int(cand.argmax())
            if cand[j] >= thr:
                tp[k] = True; taken[j] = True; gt_conf[j] = conf[order[k]]
    return conf[order], tp, gt_conf

def average_precision(conf, tp, n_gt):
    if n_gt == 0 or len(conf) == 0: return 0.0
    o = np.argsort(-conf); tp = tp[o]
    ctp, cfp = np.cumsum(tp), np.cumsum(~tp)
    rec, prec = ctp/n_gt, ctp/(ctp+cfp)
    mrec = np.concatenate([[0.], rec, [1.]])
    mpre = np.concatenate([[1.], prec, [0.]])
    mpre = np.flip(np.maximum.accumulate(np.flip(mpre)))
    x = np.linspace(0, 1, 101)
    return float(_trapz(np.interp(x, mrec, mpre), x))

def evaluate_predictions(preds, img_paths, name=''):
    gts = [gt_to_xyxy(read_yolo_labels(label_path_for(p)), pr['shape'])
           for p, pr in zip(img_paths, preds)]
    n_gt = int(sum(len(g) for g in gts))
    gt_area = np.concatenate([
        ((g[:,2]-g[:,0])*(g[:,3]-g[:,1])) / (pr['shape'][0]*pr['shape'][1])
        for g, pr in zip(gts, preds)])
    aps, keep = [], None
    for thr in IOU_THRS:
        confs, tps, gtc = [], [], []
        for pr, g in zip(preds, gts):
            c, t, gc = match_image(pr['xyxy'], pr['conf'], g, thr)
            confs.append(c); tps.append(t); gtc.append(gc)
        aps.append(average_precision(np.concatenate(confs), np.concatenate(tps), n_gt))
        if abs(thr - 0.5) < 1e-9: keep = (confs, tps, gtc)
    confs, tps, gtc = keep
    return {'name': name, 'n_images': len(img_paths), 'n_gt': n_gt,
            'ap50': aps[0], 'ap5095': float(np.mean(aps)),
            'conf50': np.concatenate(confs), 'tp50': np.concatenate(tps),
            'gt_conf50': np.concatenate(gtc), 'gt_area': gt_area,
            'img_conf': confs, 'img_tp': tps,
            'img_ngt': [len(g) for g in gts]}

def op_point(res, thr):
    sel = res['conf50'] >= thr
    tp = int(res['tp50'][sel].sum()); fp = int(sel.sum()) - tp
    fn = res['n_gt'] - tp
    P = tp/max(tp+fp,1); R = tp/max(res['n_gt'],1)
    return {'conf': float(thr), 'P': P, 'R': R,
            'F1': 2*P*R/(P+R+1e-9), 'TP': tp, 'FP': fp, 'FN': fn}

def best_f1_conf(res, grid=np.linspace(0.02,0.95,94)):
    f1 = [op_point(res, t)['F1'] for t in grid]
    return float(grid[int(np.argmax(f1))]) if max(f1) > 0 else 0.25

def recall_by_size(res, thr):
    out = {}
    for lbl,(lo,hi) in {'small (<1%)': (0,0.01),
                         'medium (1-10%)': (0.01,0.10),
                         'large (>=10%)': (0.10,9)}.items():
        m = (res['gt_area'] >= lo) & (res['gt_area'] < hi)
        out[lbl] = (int(m.sum()),
                    float((res['gt_conf50'][m] >= thr).mean()) if m.any() else float('nan'))
    return out

print('Evaluation helpers loaded.')

## 9. Results — 3×3 Transfer Matrix

Every trained model is evaluated on **all three test sets** (day, night-synthetic, real-night). Confidence thresholds are selected per model on its own-domain validation set. Results are averaged over seeds with ± std.

In [ ]:
RES, THR = {}, {}
for seed, pre, dom in tqdm(done, desc='Evaluating'):
    model = YOLO(str(best_path(pre, dom, seed)))
    # Evaluate on all three test domains
    for tdom in DOMAINS:
        paths = BENCH_SETS[(pre, tdom, 'test')]['images']
        RES[(pre, dom, seed, tdom)] = evaluate_predictions(
            predict_dataset(model, paths), paths, f'{dom}-trained | {tdom} test')
    # Choose threshold on own-domain val set
    vp = BENCH_SETS[(pre, dom, 'val')]['images']
    THR[(pre, dom, seed)] = best_f1_conf(
        evaluate_predictions(predict_dataset(model, vp), vp, 'val'))
    del model; gc.collect(); torch.cuda.empty_cache()

rec = []
for (pre, dom, seed, tdom), r in RES.items():
    op = op_point(r, THR[(pre, dom, seed)])
    rec.append({'pre': pre, 'train': dom, 'seed': seed, 'test': tdom,
                'mAP50': r['ap50'], 'mAP50-95': r['ap5095'],
                'conf': op['conf'], 'P': op['P'], 'R': op['R'],
                'F1': op['F1'], 'TP': op['TP'], 'FP': op['FP'], 'FN': op['FN']})
df_res = pd.DataFrame(rec)

def mean_std(x): return f'{x.mean():.3f} ± {x.std(ddof=0):.3f}'
for metric in ('mAP50', 'mAP50-95'):
    print(f'\n===== {metric}: rows = training data, cols = test set (mean ± std over {len(SEEDS)} seeds) =====')
    for pre in PREPROCS:
        sub = df_res[df_res.pre == pre]
        tab = sub.pivot_table(index='train', columns='test', values=metric,
                              aggfunc=mean_std).reindex(index=DOMAINS, columns=DOMAINS)
        print(f'\n[{pre}]'); display(tab)

In [ ]:
# ---- Claim quantification with bootstrap CIs ----
def boot_ap50(res_list, idx):
    vals = []
    for r in res_list:
        conf = np.concatenate([r['img_conf'][i] for i in idx])
        tp   = np.concatenate([r['img_tp'][i]   for i in idx])
        vals.append(average_precision(conf, tp,
                    int(sum(r['img_ngt'][i] for i in idx))))
    return float(np.mean(vals))

brng = np.random.default_rng(SEED)
claim_rows = []
for pre in PREPROCS:
    L = {dom: [RES[(pre,dom,s,'real')] for s in SEEDS
                if (pre,dom,s,'real') in RES] for dom in DOMAINS}
    if not all(L.values()): continue
    n_img = L['night'][0]['n_images']
    d_gain = []
    for _ in range(N_BOOT):
        idx = brng.integers(0, n_img, n_img)
        a_d, a_n = boot_ap50(L['day'],idx), boot_ap50(L['night'],idx)
        d_gain.append(a_n - a_d)
    ap = {d: np.mean([r['ap50'] for r in L[d]]) for d in DOMAINS}
    lo, hi = np.percentile(d_gain, [2.5, 97.5])
    claim_rows.append({'preprocessing': pre,
        'Day → real night (mAP50)': ap['day'],
        'Night-synthetic → real night': ap['night'],
        'Night-real → real night (ceiling)': ap['real'],
        'transformation gain': ap['night'] - ap['day'],
        'gain 95% CI': f'[{lo:.3f}, {hi:.3f}]',
        'P(gain > 0)': float(np.mean(np.array(d_gain) > 0)),
        'ceiling recovered (night)': ap['night'] / max(ap['real'], 1e-9),
        'ceiling recovered (day)':   ap['day']   / max(ap['real'], 1e-9)})
claims = pd.DataFrame(claim_rows).set_index('preprocessing')
display(claims.round(3).T)

In [ ]:
# Grouped bar chart: AP50 of each training domain on each test set
fig, ax = plt.subplots(1, len(PREPROCS), figsize=(7*len(PREPROCS), 4.8), squeeze=False)
for k, pre in enumerate(PREPROCS):
    sub = df_res[df_res.pre == pre]
    for j, dom in enumerate(DOMAINS):
        g = sub[sub.train == dom].groupby('test')['mAP50']
        m_ = g.mean().reindex(DOMAINS); s_ = g.std(ddof=0).reindex(DOMAINS)
        b = ax[0][k].bar(np.arange(3)+(j-1)*0.27, m_.values, 0.27,
                         yerr=s_.values, capsize=3,
                         label=f'trained on {dom}', color=cols[dom])
        ax[0][k].bar_label(b, fmt='%.2f', fontsize=8)
    ax[0][k].set_xticks(range(3))
    ax[0][k].set_xticklabels([f'{d} test' for d in DOMAINS])
    ax[0][k].set_ylim(0, 1); ax[0][k].set_title(f'mAP@0.5 [{pre}]')
    ax[0][k].legend(fontsize=8)
plt.tight_layout(); savefig('transfer_matrix.png'); plt.show()

In [ ]:
# PR curves on the REAL night test set
def pr_curve(res):
    o = np.argsort(-res['conf50']); tp = res['tp50'][o]
    ctp, cfp = np.cumsum(tp), np.cumsum(~tp)
    return ctp/max(res['n_gt'],1), ctp/np.maximum(ctp+cfp,1)

fig, ax = plt.subplots(1, len(PREPROCS), figsize=(6.5*len(PREPROCS), 5), squeeze=False)
for k, pre in enumerate(PREPROCS):
    for dom in DOMAINS:
        key = (pre, dom, SEEDS[0], 'real')
        if key in RES:
            r_, p_ = pr_curve(RES[key])
            ax[0][k].plot(r_, p_, color=cols[dom],
                          label=f'trained on {dom} (AP50={RES[key]["ap50"]:.3f})')
    ax[0][k].set_xlabel('Recall'); ax[0][k].set_ylabel('Precision')
    ax[0][k].set_xlim(0,1); ax[0][k].set_ylim(0,1.02)
    ax[0][k].set_title(f'PR @ IoU 0.5 — REAL night test [{pre}]')
    ax[0][k].grid(alpha=.3); ax[0][k].legend(fontsize=8)
plt.tight_layout(); savefig('pr_curves_real.png'); plt.show()

## 10. Diagnostics, Qualitative Results & Saved Artefacts

In [ ]:
rows = []
for (pre, dom, seed, tdom), r in RES.items():
    if tdom != 'real': continue
    for lbl, (n, rc) in recall_by_size(r, THR[(pre, dom, seed)]).items():
        rows.append({'pre': pre, 'train': dom, 'size': lbl, 'GT boxes': n, 'recall': rc})
sz = pd.DataFrame(rows).groupby(['pre','train','size']).agg({'GT boxes':'first','recall':'mean'}).round(3)
display(sz.unstack('train')['recall'].reindex(columns=DOMAINS))

In [ ]:
if RUN_IMGSZ_SWEEP:
    print('ORACLE diagnostic: uses real VAL labels. Not part of the main claim.')
    sweep = []
    for pre in PREPROCS:
        if not best_path(pre, 'night', SEEDS[0]).exists(): continue
        m_ = YOLO(str(best_path(pre, 'night', SEEDS[0])))
        vp = BENCH_SETS[(pre, 'real', 'val')]['images']
        for s in sorted({int(round(f*IMGSZ/32)*32) for f in (0.5,0.75,1.0,1.25)}):
            sweep.append({'pre': pre, 'infer_imgsz': s,
                          'mAP50(night-trained, real val)':
                              evaluate_predictions(predict_dataset(m_, vp, imgsz=s), vp)['ap50']})
        del m_; gc.collect(); torch.cuda.empty_cache()
    display(pd.DataFrame(sweep).round(3))

In [ ]:
VIS_PRE = PREPROCS[-1]
vis_models = {dom: YOLO(str(best_path(VIS_PRE, dom, SEEDS[0])))
              for dom in DOMAINS if best_path(VIS_PRE, dom, SEEDS[0]).exists()}

def show_grid(paths, fname, title):
    cols_ = 1 + len(vis_models)
    fig, axes = plt.subplots(len(paths), cols_,
                             figsize=(5*cols_, 5*len(paths)), squeeze=False)
    for i, p in enumerate(paths):
        ims = [('Ground truth',
                draw_boxes(cv2.imread(str(p)), read_yolo_labels(label_path_for(p))))]
        for dom, m in vis_models.items():
            c = THR[(VIS_PRE, dom, SEEDS[0])]
            pred_img = m.predict(str(p), conf=c, imgsz=IMGSZ,
                                  device=DEVICE, verbose=False)[0].plot(line_width=2)
            ims.append((f'{dom}-trained (conf {c:.2f})', pred_img))
        for j, (t, im) in enumerate(ims):
            axes[i][j].imshow(cv2.cvtColor(im, cv2.COLOR_BGR2RGB))
            axes[i][j].axis('off'); axes[i][j].set_title(t, fontsize=9)
    plt.suptitle(title, y=1.0); plt.tight_layout(); savefig(fname); plt.show()

real_test_imgs = BENCH_SETS[(VIS_PRE, 'real', 'test')]['images']
show_grid(random.Random(24).sample(real_test_imgs, min(5, len(real_test_imgs))),
          'qualitative_real.png', f'Real night test [{VIS_PRE}]')

night_test_imgs = BENCH_SETS[(VIS_PRE, 'night', 'test')]['images']
show_grid(random.Random(SEED).sample(night_test_imgs, min(3, len(night_test_imgs))),
          'qualitative_night_synth.png', f'Synthetic night test [{VIS_PRE}]')

del vis_models; gc.collect(); torch.cuda.empty_cache()

In [ ]:
print('=' * 78)
for pre, r in claims.iterrows():
    print(f'[{pre}]')
    print(f'  Trained on DAY images (New Dataset)      → real night mAP50 = {r["Day → real night (mAP50)"]:.3f}')
    print(f'  Trained on TRANSFORMED NIGHT images      → real night mAP50 = {r["Night-synthetic → real night"]:.3f}')
    print(f'  Trained on REAL NIGHT images (ceiling)   → real night mAP50 = {r["Night-real → real night (ceiling)"]:.3f}')
    print(f'  Transformation gain = {r["transformation gain"]:+.3f}  95% CI {r["gain 95% CI"]}  P(gain>0) = {r["P(gain > 0)"]:.2f}')
    print(f'  Synthetic night recovers {100*r["ceiling recovered (night)"]:.0f}% of the real-night ceiling '
          f'(day-trained recovers {100*r["ceiling recovered (day)"]:.0f}%)')
print('=' * 78)

In [ ]:
df_res.round(4).to_csv(WORK/'results_long.csv', index=False)
claims.round(4).to_csv(WORK/'results_claims.csv')
sz.to_csv(WORK/'results_recall_by_size.csv')
bal.to_csv(WORK/'benchmark_balance.csv')

summary = {
    'dataset': 'New Dataset (VOC XML → YOLO converted)',
    'config': {'model': MODEL_WEIGHTS, 'imgsz': IMGSZ, 'epochs': EPOCHS,
               'batch': BATCH, 'seeds': SEEDS, 'preprocs': PREPROCS,
               'target_mean': TARGET_MEAN, 'clahe_clip': CLAHE_CLIP,
               'hash_thr': HASH_THR, 'split_fracs': SPLIT_FRACS,
               'smoke_test': SMOKE_TEST},
    'sizes': {'train': N_TRAIN, 'val': N_VAL, 'test': N_TEST},
    'conversion': {'total_xml': len(list(ANNO_DIR.glob('*.xml'))),
                   'converted': len(all_items)},
    'thresholds': {f'{k[1]}|{k[0]}|s{k[2]}': v for k, v in THR.items()},
    'claims': claims.round(4).reset_index().to_dict('records')
}
(WORK/'results_summary.json').write_text(json.dumps(summary, indent=2, default=str))

ZIP_PATH = WORK / 'report_outputs_new_dataset.zip'
ZIP_PATH.unlink(missing_ok=True)
include = [WORK/f for f in ('results_long.csv','results_claims.csv',
                             'results_recall_by_size.csv','benchmark_balance.csv',
                             'results_summary.json')]
include += sorted(FIGS.glob('*')) + sorted(BENCH.glob('*.yaml'))
include += [best_path(pre, dom, SEEDS[0])
             for pre in PREPROCS for dom in DOMAINS
             if best_path(pre, dom, SEEDS[0]).exists()]
include += [RUNS/run_name(pre,dom,s)/'results.csv'
             for pre in PREPROCS for dom in DOMAINS for s in SEEDS
             if (RUNS/run_name(pre,dom,s)/'results.csv').exists()]
with zipfile.ZipFile(ZIP_PATH, 'w', zipfile.ZIP_DEFLATED) as z:
    for p in include:
        if Path(p).exists():
            z.write(p, Path(p).relative_to(WORK))
print(f'ZIP: {ZIP_PATH} | {ZIP_PATH.stat().st_size/1e6:.1f} MB')
display(FileLink('report_outputs_new_dataset.zip',
                 result_html_prefix='Download all results: '))